In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_ollama import ChatOllama,OllamaEmbeddings
from typing import TypedDict, Literal,List,Dict,Annotated
from langgraph.graph import StateGraph,START,END,MessagesState
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage,BaseMessage,ToolMessage
import operator
from langchain_core.tools import tool
import uuid
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command,interrupt
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware,PIIMiddleware,SummarizationMiddleware
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.tools import ToolException
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import BM25Retriever,EnsembleRetriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain


In [ ]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=200)
llm = ChatBedrockConverse(model='claude_haiku',region_name='us-east-1',aws_access_key_id='',aws_secret_access_key='')
embedding_model = OllamaEmbeddings(model="nomic-embed-text")


Streaming disabled for model 'claud_haiku': provider 'claud_haiku' is not verified as streaming-capable with ChatBedrockConverse, so calls will fall back to the non-streaming Converse API. If this model does support streaming, pass disable_streaming=False to override.


In [ ]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]
    question: str

In [ ]:
graph = StateGraph(State)
graph.add_node('classify_query_node',classify_query)
graph.add_node('explain_node',explain_node)
graph.add_node('opinion_node',opinion_node)
graph.add_node('guide_node',guide_node)

graph.add_edge(START,'classify_query_node')
graph.add_conditional_edges(
    'classify_query_node',
    check_condition,   # we can also use lambda function here
    {
        'explain_node':'explain_node',
        'opinion_node':'opinion_node'
    }
)
graph.add_edge('explain_node',END)
graph.add_edge('opinion_node',END)
graph.add_edge('guide_node',END)

# 1 way
conn = sqlite3.connect("database/data.db",check_same_thread=False)
app = graph.compile(checkpointer=SqliteSaver(conn=conn))

# 2 way
store = InMemoryStore()
app = graph.compile(store=store)

# 3 way
app = graph.compile(checkpointer=SqliteSaver(conn=conn),store=store)

In [ ]:
from IPython.display import display, Image

# 1
print(app.get_graph().draw_ascii())
# 2
display(Image(app.get_graph().draw_mermaid_png()))


In [ ]:
thread_id = uuid.uuid4()
config = {'configurable':{'thread_id':thread_id}}
res = app.invoke({'query':'How to buid a langgraph'},config=config)

human_response = interrupt({'message_for_user':"approve kar be"})

# Resume the excecution if interrupted
res = app.invoke(Command(resume="You can execute now"),config=config)

In [ ]:
### RAG evaluation 

## Retrieval evaluation metrix ---> 
# context precision
# context recall
# F1 score
# Mean reciprocal Rank 
# Normalized discounted cumulative gain

## Generation evaluation metrix --->
# BLUE
# ROUGH
# Faithfulness
# Answer relevancy
# Answer correctness


In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

prompt = ChatPromptTemplate.from_messages([
    ('system','give answers based on below context : \n {context}'),
    ('human','question : {question}')
])

chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
rchain = create_retrieval_chain(retriever=retriever,combine_docs_chain=chain)

res = rchain.invoke({'question':'what is the leave policy for hexa ai company'})

SyntaxError: expected argument value expression (1991794806.py, line 1)

In [ ]:
# Types of RAG 
# 1] Agentic RAG routers: dynamic rask routing and integration with retrieval/generation 
# 2] Query planner Agentic RAG: decomposing complex queries and parallelizing subqueries 
# 3] Corrective RAG Agents: Error detection,feedback loops and self-correction
# 4] Adaptive RAG Agents: Learning from interactions and adjusting retrieval strategies
# 5] Speculative RAG: pedictive task execution and uncertainty

